程序已经跑通，这意味着你已经完成了从“单细胞数据筛选”到“空间探针生成”的全过程。在你的服务器目录结构中，这些结果通常分布在两个不同的位置：一个是靶点列表（Panel），一个是序列设计（Probes）。

按照我们之前的代码逻辑，你可以通过以下方式找到它们：

📂 1. Panel 在哪里？（基因名单）
Panel 是你在第一阶段（R 脚本）根据 scRNA-seq 差异表达分析筛选出的基因集。

文件位置：/home/zhen/GZ_Projects_2026/06_Visium_Analysis/custom_panel_targets.csv


cd /home/zhen/GZ_Projects_2026/06_Visium_Analysis/
head -n 20 custom_panel_targets.csv
它长什么样：这是一张包含基因名称（Gene Name）、所属细胞簇（Cluster）和平均倍数变化（avg_log2FC）的表格。


🧬 2. Probe 在哪里？（探针序列）
Probe 是你在第二阶段（Python 脚本）针对上述 Panel 基因生成的具体碱基序列。

文件位置：如果你运行了完整的自动化流，它通常会生成一个 .fasta 或 .csv 文件，命名为 final_probes_set.csv（或类似的名称）。

结构解析：
一个标准的空间转录组探针（如 Xenium 或 MERFISH 类型）通常由三部分组成：

靶向序列 (Targeting Sequence)：与 mRNA 互补的部分（约 25-30 bp）。
连接序列 (Linkers)：中间的结构支撑。
Readout 序列 (条形码部分)：与荧光探针结合的部分。

💻 命令行快速定位
如果你不确定具体的输出文件名，可以使用 find 命令在工作目录中搜索：




# 激活环境并进入目录
mamba activate R44_RNA
cd /home/zhen/GZ_Projects_2026/06_Visium_Analysis/

# 搜索包含 "panel" 或 "probe" 字样的 csv 文件
ls -lh *panel* *probe*
📊 结果核对清单 (Raw Markdown)
你可以将此表复制到 VS Code，对照检查你的输出是否完整：



Markdown
| 组件名称 | 物理文件 | 核心作用 | 解决的问题 |
| :--- | :--- | :--- | :--- |
| **Custom Panel** | `custom_panel_targets.csv` | 确定要检测的 300-500 个基因 | 挑选细胞类型特异性 Marker |
| **Probe List** | `final_probes_set.csv` | 包含所有探针的碱基序列 (A/T/C/G) | 实现原位杂交 (ISH) |
| **Barcode Map**| `barcode_assignment.json` | 记录哪个基因对应哪种颜色组合 | 解决 **光学条形码** 解码问题 |
| **QC Stats** | `probe_quality_report.txt`| 记录 Tm 值和 GC 含量 | 避免 **Optical Crowding** 和非特异结合 |

设计针对单细胞分辨率空间转录组（如 Xenium 或 MERFISH）的探针，
是一项将分子生物学和生物信息学紧密结合的精密工程。这类技术的探针设计与传统的非原位测序探针有显著区别，  
它们需要同时解决靶标特异性结合和光学条形码解码两个大问题。

以下是为你整理的亚细胞级空间转录组探针设计的完整技术解析。

📐 1. 探针标准结构与长度在 Xenium 或 MERFISH 中，一个标准的探针（Encoding Probe）总长度通常在 70 到 120 nt（核苷酸） 之间。它的典型模块化结构如下： 

靶向序列 (Targeting Sequence, ~30-50 nt)： 
作用：直接与目标组织中的 mRNA 互补配对。 
Xenium 的特殊设计（Padlock/挂锁探针）：为了极高的特异性，Xenium 会将这段靶向序列一分为二（Left 和 Right，各约 20-25 nt）。只有当两端完美结合在目标 mRNA 上且没有错配时，连接酶才能将其连成环状，随后进行滚环扩增（RCA）。 

读出序列 (Readout Sequences, ~15-20 nt / 个)： 
作用：这些是不存在于生物体内的“外星”序列（正交序列）。它们悬挂在靶标外部，用来与带有荧光基团的短探针（Reporter Probes）结合。 
数量：通常一个探针会带有 2 到 3 个 Readout 区域，用于实现组合条形码的多次荧光成像解码。 

连接/引物序列 (Spacers/Primers, 极短)：用于在探针合成或文库构建时提供物理缓冲或扩增结合点。

🧬 2. 探针设计要点（以神经肌肉疾病为例）在神经肌肉系统（如运动神经元、骨骼肌切片）中设计探针，面临着极高的挑战：  

高度同源基因的区分 (Isoform Specificity)： 
骨骼肌中有大量的肌球蛋白重链（MYH）家族基因（如代表慢肌的 MYH7 和快肌的 MYH2）。它们序列高度同源。靶向序列必须设计在这些转录本中突变差异最大的外显子连接处或 UTR 区域，避免探针“串门”。  

避免脱靶 (Off-target filtering)： 
必须将设计好的 30-50 nt 序列放到转录组数据库（如小鼠或人类全转录本）中进行 BLAST，剔除那些能与非目标基因发生 15 nt 以上连续匹配的序列。  

理化性质控制 (Tm 与 GC 含量)： 
GC 含量：严格控制在 40% - 60% 之间。肌肉组织富含结缔组织和肌纤维，如果探针 GC 太高，极易产生非特异性粘附形成背景噪音。 
Tm 值（解链温度）：Panel 中所有 300 个探针的 Tm 值必须保持在一个极小的区间内（例如 65°C ± 2°C），以保证它们在同一实验温度下都能完美杂交。 

🔬3. 真实探针设计实例（神经肌肉核心靶点）下面展示几个针对神经肌肉疾病（如肌萎缩侧索硬化症 ALS 或杜氏肌营养不良 DMD）的虚拟探针结构实例：探针实例结构表 (附 Raw Markdown)

| 靶点基因 (细胞类型) | 探针模块划分示例 (5' -> 3') | 靶向序列 (模拟) | Readout 组合设定 |
| :--- | :--- | :--- | :--- |
| **DMD** (骨骼肌纤维) | Readout1 - Spacer - Target_L - Target_R - Readout2 | `GACATGC...TTGATCG` | 绑定荧光通道 1 & 4 |
| **CHAT** (运动神经元) | Readout3 - Target_L - Target_R - Readout5 | `CCGTAAA...GGCCTTA` | 绑定荧光通道 2 & 3 |
| **GFAP** (星形胶质细胞) | Readout1 - Target_L - Target_R - Readout5 | `ATGCTAG...CCATTGA` | 绑定荧光通道 1 & 3 |


📂 4. 一次实验获得的结果是怎样的？ 

实验完成后，仪器的核心计算节点会吐出一组数据。它不像 Visium 那样基于 55 微米的圆点，而是真正到单个转录本（分子）级别。 

你会得到以下几个核心文件：  
transcripts.parquet (或 csv)：极其庞大的表格，记录了检测到的每一个 mRNA 分子的 X, Y, Z 物理坐标及其对应的基因名称和质量分数。
cells.csv：基于细胞核形态（DAPI）和细胞膜标志物勾勒出的细胞多边形边界坐标。 cell_feature_matrix.h5：类似于你单细胞分析时的表达矩阵（行是基因，列是细胞），可以直接导入 Seurat V5 进行降维聚类。形态学高分辨 TIFF 图像：带有 DAPI 和组织自发荧光的极高分辨率大图。

💰 5. 实验价格与分析周期基于目前的行业标准（2026年）：实验成本（试剂与芯片）：购买官方预制 Panel（如全脑或神经测序）：单张载玻片（通常可放 2-4 个小组织）费用约在 $1,500 - $2,500 美元之间。定制 Panel（你自己用 custom_panel_targets.csv 提交的基因集）：通常需要支付额外的一笔数千美元的探针合成起步费，折合到单个样品的测试成本可能会上升。分析周期：上机与化学循环：约 2 到 4 天（取决于读取的基因数量和组织面积）。生物信息学分析：拿到数据后，在你的 Linux 服务器上进行环境配置、数据过滤、细胞分割校正（Segmentation Refinement）到生成降维聚类图，一个熟练的生信分析师通常需要 1 到 2 周。💻 服务器演示代码你可以用以下代码在你的服务器上创建一个模拟 Xenium 结果的挂载点，感受一下数据结构的层级：Bash# 请先开启 screen 避免操作中断
screen -S xenium_demo
cd /home/gao/Code/Bioinfo_Analysis_Projects
mamba activate R44_RNA

# 创建一个模拟的 Xenium 结果输出目录
mkdir -p 07_Xenium_Analysis/sample_muscle_01/
cd 07_Xenium_Analysis/sample_muscle_01/

# 模拟仪器下机生成的核心文件列表
touch transcripts.parquet cells.csv cell_feature_matrix.h5 morphology_focus.ome.tif
ls -lh
我们之前已经帮你生成了 custom_panel_targets.csv。下一步，我们要不要在 VS Code 里用 Python 写一个小脚本，提取你这个 Panel 中所有基因的参考 mRNA 序列，并计算它们的 GC 含量，看看它们是否符合 40%-60% 的探针设计黄金标准？